# Toto-2 — Comparativa de calendarios (testing)

Réplica del experimento del notebook **DSA**, pero con el modelo fundacional **Toto-2** (Datadog) como motor de predicción *zero-shot*.

Entrada: un **Google Spreadsheet** (o CSV) con las columnas `timestamp` y `target` (dato bruto en niveles).

| | Escenario | `week_days` | `weekend_fill` |
|---|-----------|-------------|----------------|
| (a) | Calendario completo (7 días), forecast como siempre | 7 | — |
| (b) | Calendario de negocio (5 días): se eliminan los fines de semana y se rellenan con la **media de la semana anterior** (5 días hábiles) | 5 | `"mean"` |
| (c) | Calendario de negocio (5 días): se eliminan los fines de semana y se rellenan por **interpolación lineal** (viernes→lunes) | 5 | `"interp"` |

Salida: **métricas desagregadas** en un archivo `toto2_univariate_{escenario}.xlsx` por escenario, con las hojas `metadata` (resumen del experimento y de las ventanas), `predictions` (real vs predicho por ventana, en niveles) y `statistics` (resultados de la evaluación).

> **Nota sobre covariables.** Toto-2.0 es un forecaster *zero-shot* univariante/multivariante que **aún no admite covariables exógenas** (regresores conocidos a futuro). Por eso las covariables de calendario se calculan (para mantener la estructura del experimento y la metadata) pero **no** se pasan al modelo: la información de calendario entra únicamente a través de la construcción de la serie (el relleno de fines de semana del calendario de negocio).

> **Validación rápida.** El notebook viene configurado para una prueba sencilla de **2 ventanas** con **Toto-2.0-4M**. Sube `N_FORECAST_WINDOWS` y/o cambia `TOTO_MODEL_SIZE` para el experimento completo o para modelos mayores.

## (1) Paquetes

In [ ]:
# ============================================================
# Instalación de paquetes (en Colab). Toto-2 requiere Python >= 3.12 y
# PyTorch >= 2.5; se recomienda GPU (Ampere o superior), aunque corre en CPU.
# ============================================================
%pip install -q toto-models                       # Toto-2 (Datadog) -> from toto2 import Toto2Model
%pip install -q pandas numpy openpyxl matplotlib gspread

In [ ]:
# ============================================================
# Importar librerías
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from toto2 import Toto2Model

print("torch", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())

## (2) Definición de funciones

In [ ]:
# ============================================================
# DEFINICIÓN DE FUNCIONES (vendimias / growing-window forecast con Toto-2)
# ============================================================
# Mismo experimento que el notebook DSA, pero el motor de predicción es el
# modelo fundacional Toto-2 (Datadog). Toto-2 es un forecaster ZERO-SHOT
# univariante/multivariante: NO admite (todavía) covariables exógenas, por lo
# que la información de calendario entra SÓLO a través de la construcción de la
# serie (relleno de fines de semana en el calendario de negocio), no como
# regresores. Las covariables de calendario se calculan igualmente para
# mantener la estructura del experimento y la metadata.
# ============================================================
import numpy as np
import pandas as pd

# ---- Transformación y su inversa (log / log1p / level) ----
def transform_level(x, transformation="log"):
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy()
    if transformation == "log":
        return np.log(x)
    if transformation == "log1p":
        return np.log1p(x)
    if transformation == "level":
        return x
    raise ValueError("transformation debe ser 'log', 'log1p' o 'level'.")

def inverse_transform(x, transformation="log"):
    x = np.asarray(x, dtype="float64")
    if transformation == "log":
        return np.exp(x)
    if transformation == "log1p":
        return np.expm1(x)
    if transformation == "level":
        return x
    raise ValueError("transformation debe ser 'log', 'log1p' o 'level'.")

# ---- Domingo de Pascua (Meeus/Jones/Butcher), sin dependencias externas ----
def _easter_sunday(year):
    a = year % 19
    b = year // 100
    c = year % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i = c // 4
    k = c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    return pd.Timestamp(year=year, month=month, day=day)

# ============================================================
# Covariables de calendario (se calculan; Toto-2 no las usa como regresores)
# ============================================================
def add_calendar_covariates(df, date_col="timestamp"):
    df = df.copy()
    ts = pd.to_datetime(df[date_col])
    df["timestamp"] = ts
    df["dow"] = ts.dt.dayofweek                      # 0=Lun ... 6=Dom
    df["month"] = ts.dt.month
    df["day"] = ts.dt.day
    df["week"] = ts.dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["dow"] >= 5).astype(int)

    years = sorted(ts.dt.year.unique())
    fixed = []
    for y in years:
        for mmdd in ["01-01", "01-06", "05-01", "05-02", "06-24", "08-15",
                     "10-12", "11-01", "12-06", "12-08", "12-25", "12-31"]:
            fixed.append(pd.Timestamp(f"{y}-{mmdd}"))
    easter_related = []
    for y in years:
        e = _easter_sunday(y)
        for off in range(-7, 2):
            easter_related.append(e + pd.Timedelta(days=off))
    easter_related = pd.DatetimeIndex(easter_related)
    holidays_all = pd.DatetimeIndex(fixed).union(easter_related)

    tsn = ts.dt.normalize()
    df["is_holiday"] = tsn.isin(holidays_all).astype(int)
    df["is_pre_holiday"] = (tsn + pd.Timedelta(days=1)).isin(holidays_all).astype(int)
    df["is_post_holiday"] = (tsn - pd.Timedelta(days=1)).isin(holidays_all).astype(int)
    df["is_bridge_day"] = (((df["is_weekend"] == 0) & (df["is_holiday"] == 0)) &
                           ((df["is_pre_holiday"] == 1) | (df["is_post_holiday"] == 1))).astype(int)
    df["is_easter_week"] = tsn.isin(easter_related).astype(int)
    return df

# ---- Secuencia de fechas respetando el calendario elegido ----
def date_sequence_by_week_days(start_date, end_date, week_days=7):
    start_date = pd.Timestamp(start_date).normalize()
    end_date = pd.Timestamp(end_date).normalize()
    if pd.isna(start_date) or pd.isna(end_date) or start_date > end_date:
        return pd.DatetimeIndex([])
    idx = pd.date_range(start_date, end_date, freq="D")
    if week_days == 5:
        idx = idx[idx.dayofweek <= 4]
    elif week_days != 7:
        raise ValueError("week_days debe ser 5 (lunes-viernes) o 7 (calendario completo).")
    return idx

# ---- ¿Contiene el rango [a, b] algún 29 de febrero? (Tarea 2) ----
def range_has_leap_day(a, b):
    a = pd.Timestamp(a).normalize(); b = pd.Timestamp(b).normalize()
    if pd.isna(a) or pd.isna(b) or a > b:
        return False
    d = pd.date_range(a, b, freq="D")
    return bool(((d.month == 2) & (d.day == 29)).any())

# ---- n-ésimo día hábil del mes (cortes en calendario de negocio) ----
def nth_business_day_of_month(month_start, n):
    month_start = pd.Timestamp(month_start).normalize()
    ms = month_start.replace(day=1)
    me = (ms + pd.offsets.MonthEnd(0)).normalize()
    days = pd.date_range(ms, me, freq="D")
    bdays = days[days.dayofweek <= 4]
    if n > len(bdays):
        return pd.NaT
    return bdays[n - 1]

# ---- Agregación de un vector de niveles ----
def aggregate_level(x, agg="sum", empty_value=np.nan):
    x = np.asarray(x, dtype="float64")
    x = x[np.isfinite(x)]
    if x.size == 0:
        return empty_value
    if agg == "sum":
        return float(np.sum(x))
    if agg == "mean":
        return float(np.mean(x))
    raise ValueError("agg debe ser 'sum' o 'mean'.")

# ============================================================
# Relleno de fines de semana para el calendario de negocio (week_days = 5)
# ------------------------------------------------------------
#   "mean"   : media de los 5 días hábiles (lun-vie) inmediatamente anteriores
#   "interp" : interpolación lineal entre el último viernes y el lunes siguiente
# Trabaja sobre la serie en NIVELES; el caller recalcula la serie transformada.
# ============================================================
def fill_weekends(df, date_col, level_col, mode="mean"):
    if mode not in ("mean", "interp"):
        raise ValueError("mode debe ser 'mean' o 'interp'.")
    df = df.sort_values(date_col).reset_index(drop=True).copy()
    dts = pd.to_datetime(df[date_col])
    is_wknd = (dts.dt.dayofweek >= 5).to_numpy()
    lvl = pd.to_numeric(df[level_col], errors="coerce").to_numpy(dtype="float64")

    if mode == "interp":
        s = pd.Series(lvl, index=pd.DatetimeIndex(dts))
        s[is_wknd] = np.nan
        s = s.interpolate(method="time", limit_direction="both")
        lvl_filled = s.to_numpy()
    else:  # "mean"
        bus_idx = np.where(~is_wknd)[0]
        lvl_filled = lvl.copy()
        for i in np.where(is_wknd)[0]:
            prev_bus = bus_idx[bus_idx < i]
            if prev_bus.size >= 1:
                lvl_filled[i] = np.nanmean(lvl[prev_bus[-5:]])
            else:
                nxt_bus = bus_idx[bus_idx > i]
                lvl_filled[i] = np.nanmean(lvl[nxt_bus[:5]]) if nxt_bus.size else np.nan

    # Cierra cualquier hueco restante (p.ej. festivos no hábiles) de forma robusta
    s = pd.Series(lvl_filled, index=pd.DatetimeIndex(dts)).interpolate(
        method="time", limit_direction="both")
    s = s.ffill().bfill()
    df[level_col] = s.to_numpy()
    return df

# ============================================================
# Llamada al modelo Toto-2 sobre una serie de contexto univariante (PROC/log)
# Devuelve (mediana, p10, p90) como vectores numpy de longitud `horizon`.
# El motor (torch) se importa de forma perezosa para no exigirlo si no se usa.
# ============================================================
def toto_forecast_proc(model, y_context, horizon, device="cpu",
                       decode_block_size=None, max_context=None):
    import torch
    y = np.asarray(y_context, dtype="float32")
    y = y[np.isfinite(y)]
    if max_context is not None and y.shape[0] > max_context:
        y = y[-max_context:]
    target = torch.tensor(y, dtype=torch.float32, device=device).reshape(1, 1, -1)
    target_mask = torch.ones_like(target, dtype=torch.bool)
    series_ids = torch.zeros(1, 1, dtype=torch.long, device=device)
    inputs = {"target": target, "target_mask": target_mask, "series_ids": series_ids}
    fkw = {}
    if decode_block_size is not None:
        fkw["decode_block_size"] = decode_block_size
    with torch.no_grad():
        q = model.forecast(inputs, horizon=int(horizon), **fkw)   # (9, 1, 1, horizon)
    q = q.detach().to("cpu").float().numpy()
    return q[4, 0, 0, :].copy(), q[0, 0, 0, :].copy(), q[8, 0, 0, :].copy()

# ============================================================
# Agregación de resultados diarios a semanal/mensual
# (Diario PROC -> Diario niveles -> Agregado niveles -> Agregado PROC)
# Marca además los periodos que contienen un 29 de febrero (Tarea 2).
# ============================================================
def build_aggregate_eval(daily_eval_df, context_df, date_col, level_target_col,
                         unit, week_days, aggregate_fun, transformation):
    if daily_eval_df is None or len(daily_eval_df) == 0:
        return pd.DataFrame()

    ctx = pd.DataFrame({
        ".fecha": pd.to_datetime(context_df[date_col]),
        ".level": pd.to_numeric(context_df[level_target_col], errors="coerce"),
    }).dropna(subset=[".fecha"])
    if week_days == 5:
        ctx = ctx[pd.DatetimeIndex(ctx[".fecha"]).dayofweek <= 4]

    d = daily_eval_df.copy()
    d["fecha"] = pd.to_datetime(d["fecha"])
    d["cutoff_date"] = pd.to_datetime(d["cutoff_date"])
    if "id_ventana" not in d.columns:
        d["id_ventana"] = "Corte " + d["cutoff_date"].dt.strftime("%Y-%m-%d")

    if unit == "week":
        d["period_start"] = d["fecha"].dt.to_period("W-SUN").dt.start_time
        d["period_end"] = d["period_start"] + pd.Timedelta(days=6)
    else:
        d["period_start"] = d["fecha"].dt.to_period("M").dt.start_time
        d["period_end"] = d["fecha"].dt.to_period("M").dt.end_time.dt.normalize()

    keys = ["id_ventana", "cutoff_date", "cutoff_day", "week_no", "period_start", "period_end"]
    rows = []
    for key_vals, g in d.groupby(keys, sort=True):
        g = g.sort_values("fecha")
        id_ventana, cutoff_date, cutoff_day, week_no, period_start, period_end = key_vals
        cutoff_date = pd.Timestamp(cutoff_date)
        period_start = pd.Timestamp(period_start); period_end = pd.Timestamp(period_end)
        period_dates = date_sequence_by_week_days(period_start, period_end, week_days)

        obs_full = ctx[ctx[".fecha"].isin(period_dates)]
        obs_partial = obs_full[obs_full[".fecha"] <= cutoff_date]
        available_dates = pd.DatetimeIndex(obs_full.loc[np.isfinite(obs_full[".level"]), ".fecha"].unique())
        has_full = (len(period_dates) > 0) and bool(pd.DatetimeIndex(period_dates).isin(available_dates).all())

        forecast_future_values = pd.to_numeric(g["yhat_level"], errors="coerce").to_numpy()
        observed_partial_values = obs_partial[".level"].to_numpy()

        real_level = aggregate_level(obs_full[".level"].to_numpy(), aggregate_fun) if has_full else np.nan
        pred_level = aggregate_level(np.concatenate([observed_partial_values, forecast_future_values]),
                                     aggregate_fun, empty_value=np.nan)

        real_proc = float(transform_level([real_level], transformation)[0]) if np.isfinite(real_level) else np.nan
        pred_proc = float(transform_level([pred_level], transformation)[0]) if np.isfinite(pred_level) else np.nan
        proc_error = pred_proc - real_proc

        if unit == "month":
            period_offset = (period_start.year - cutoff_date.year) * 12 + (period_start.month - cutoff_date.month)
        else:
            monday_cut = cutoff_date - pd.Timedelta(days=int(cutoff_date.dayofweek))
            period_offset = int(round((period_start - monday_cut).days / 7))

        rows.append(dict(
            id_ventana=id_ventana, cutoff_date=cutoff_date, cutoff_day=cutoff_day,
            week_no=week_no, unit=unit, period_start=period_start, period_end=period_end,
            period_offset=period_offset,
            observed_partial_level=aggregate_level(observed_partial_values, aggregate_fun, empty_value=0.0),
            forecast_future_level=aggregate_level(forecast_future_values, aggregate_fun, empty_value=np.nan),
            real_level=real_level, pred_level=pred_level, real_proc=real_proc, pred_proc=pred_proc,
            proc_error=proc_error, error_pp=100.0 * proc_error, abs_error_pp=abs(100.0 * proc_error),
            observed_partial_n=int(np.isfinite(observed_partial_values).sum()),
            forecast_n=int(np.isfinite(forecast_future_values).sum()),
            observed_full_n=int(np.isfinite(obs_full[".level"].to_numpy()).sum()),
            expected_n=len(period_dates),
            has_full_observed_period=has_full,
            has_leap_day=range_has_leap_day(period_start, period_end),
        ))
    out = pd.DataFrame(rows).sort_values(["cutoff_date", "period_start"]).reset_index(drop=True)
    return out

# ============================================================
# Predicción en vendimias (growing window) con Toto-2
#   week_days = 7 : calendario completo
#   week_days = 5 : calendario de negocio. Se ajusta sobre un calendario completo
#                   de 7 días con los fines de semana RELLENADOS según weekend_fill
#                   ("mean"/"interp"), y se evalúa SÓLO en días hábiles.
# ============================================================
def run_growing_daily_forecast(context_df, model, target_col, date_col,
                               train_history_start, forecast_start_date, last_cutoff_date,
                               cutoff_days, next_full_months=3, week_days=7,
                               weekend_fill="mean", level_target_col="target",
                               n_forecast_windows=None, aggregate_fun="sum",
                               transformation="log", device="cpu",
                               max_context=4096, decode_block_size=None):
    context_df = context_df.copy()
    context_df[date_col] = pd.to_datetime(context_df[date_col])
    train_history_start = pd.Timestamp(train_history_start)
    forecast_start_date = pd.Timestamp(forecast_start_date)
    last_cutoff_date = pd.Timestamp(last_cutoff_date)
    if target_col not in context_df.columns:
        raise ValueError(f"No existe target_col: {target_col}")
    if level_target_col not in context_df.columns:
        raise ValueError(f"No existe level_target_col: {level_target_col}")

    keep_business = (week_days == 5)

    # ---- Serie de AJUSTE: siempre calendario completo de 7 días ----
    context_fit = context_df.sort_values(date_col).reset_index(drop=True).copy()
    if week_days == 5:
        full_dates = pd.date_range(context_fit[date_col].min(), context_fit[date_col].max(), freq="D")
        base = pd.DataFrame({date_col: full_dates})
        context_fit = base.merge(context_fit[[date_col, level_target_col]], on=date_col, how="left")
        context_fit = fill_weekends(context_fit, date_col, level_target_col, weekend_fill)
        context_fit[target_col] = transform_level(context_fit[level_target_col], transformation)
    context_fit = add_calendar_covariates(context_fit, date_col=date_col)

    # context completo para la evaluación (valores reales observados)
    context_eval = context_df.sort_values(date_col).reset_index(drop=True).copy()

    max_obs_date = pd.Timestamp(context_fit[date_col].max())
    effective_last_cutoff = min(last_cutoff_date, max_obs_date)

    sequence_months = pd.date_range(forecast_start_date.to_period("M").to_timestamp(),
                                    effective_last_cutoff.to_period("M").to_timestamp(), freq="MS")

    cutoff_dates, cutoff_nominal, cutoff_weekno = [], [], []
    for m_date in sequence_months:
        for j, dd in enumerate(cutoff_days, start=1):
            if week_days == 5:
                c_date = nth_business_day_of_month(m_date, dd)
            else:
                c_date = m_date + pd.Timedelta(days=dd - 1)
            if pd.notna(c_date) and (forecast_start_date <= c_date <= effective_last_cutoff):
                cutoff_dates.append(pd.Timestamp(c_date)); cutoff_nominal.append(dd); cutoff_weekno.append(j)
    order = np.argsort([d.value for d in cutoff_dates])
    cutoff_dates = [cutoff_dates[i] for i in order]
    cutoff_nominal = [cutoff_nominal[i] for i in order]
    cutoff_weekno = [cutoff_weekno[i] for i in order]
    if n_forecast_windows is not None:
        keep = min(n_forecast_windows, len(cutoff_dates))
        cutoff_dates = cutoff_dates[:keep]; cutoff_nominal = cutoff_nominal[:keep]; cutoff_weekno = cutoff_weekno[:keep]

    fit_idx = pd.DatetimeIndex(context_fit[date_col])
    eval_lookup = context_eval.set_index(date_col)

    daily_rows, cutoff_rows = [], []
    for cutoff_date, nominal_d, week_no in zip(cutoff_dates, cutoff_nominal, cutoff_weekno):
        forecast_end = (cutoff_date.to_period("M").to_timestamp() +
                        pd.offsets.MonthBegin(next_full_months) + pd.offsets.MonthEnd(0)).normalize()

        # Horizonte sobre calendario completo (contiguo, día a día). Se predice
        # incluyendo el 29-feb para no desalinear los pasos, y se ELIMINA después.
        horizon_index = pd.date_range(cutoff_date + pd.Timedelta(days=1), forecast_end, freq="D")
        if len(horizon_index) == 0:
            continue

        train_mask = (fit_idx >= train_history_start) & (fit_idx <= cutoff_date)
        train_df = context_fit.loc[train_mask].copy()
        if len(train_df) == 0:
            continue

        y_train = pd.to_numeric(train_df[target_col], errors="coerce").to_numpy()
        median, p10, p90 = toto_forecast_proc(
            model, y_train, horizon=len(horizon_index), device=device,
            decode_block_size=decode_block_size, max_context=max_context)

        df_all = pd.DataFrame({"fecha": horizon_index, "yhat_proc": median,
                               "yhat_proc_p10": p10, "yhat_proc_p90": p90})
        # Eliminar 29-feb del horizonte (la serie NO se altera; sólo la evaluación)
        df_all = df_all[~((df_all["fecha"].dt.month == 2) & (df_all["fecha"].dt.day == 29))]
        if keep_business:
            df_all = df_all[pd.DatetimeIndex(df_all["fecha"]).dayofweek <= 4]
        df_all = df_all.reset_index(drop=True)
        if len(df_all) == 0:
            continue

        df_all["yhat_level"] = inverse_transform(df_all["yhat_proc"].to_numpy(), transformation)
        df_all["yhat_level_p10"] = inverse_transform(df_all["yhat_proc_p10"].to_numpy(), transformation)
        df_all["yhat_level_p90"] = inverse_transform(df_all["yhat_proc_p90"].to_numpy(), transformation)

        y_true_proc = eval_lookup[target_col].reindex(df_all["fecha"]).to_numpy()
        y_true_level = eval_lookup[level_target_col].reindex(df_all["fecha"]).to_numpy()

        fcst_daily = pd.DataFrame({
            "fecha": df_all["fecha"].to_numpy(),
            "yhat_proc": df_all["yhat_proc"].to_numpy(),
            "yhat_level": df_all["yhat_level"].to_numpy(),
            "yhat_level_p10": df_all["yhat_level_p10"].to_numpy(),
            "yhat_level_p90": df_all["yhat_level_p90"].to_numpy(),
            "cutoff_date": cutoff_date, "cutoff_day": nominal_d, "week_no": week_no,
            "train_start": train_df[date_col].min(), "train_end": train_df[date_col].max(),
            "forecast_start": df_all["fecha"].min(), "forecast_end": forecast_end,
            "step": np.arange(1, len(df_all) + 1), "id_ventana": f"Corte {cutoff_date.date()}",
            "y_true_proc": y_true_proc, "y_true_level": y_true_level,
        })
        fcst_daily["yhat"] = fcst_daily["yhat_proc"]
        fcst_daily["y_true"] = fcst_daily["y_true_proc"]
        fcst_daily["daily_proc_error"] = fcst_daily["yhat_proc"] - fcst_daily["y_true_proc"]
        fcst_daily["daily_error_pp"] = 100.0 * fcst_daily["daily_proc_error"]
        daily_rows.append(fcst_daily)

        cutoff_rows.append(dict(
            cutoff_date=cutoff_date, cutoff_day=nominal_d, week_no=week_no,
            train_start=train_df[date_col].min(), train_end=train_df[date_col].max(),
            forecast_start=df_all["fecha"].min(), forecast_end=forecast_end,
            prediction_length=len(df_all), train_rows=len(train_df),
            week_days=week_days, weekend_fill=(weekend_fill if week_days == 5 else None),
            context_used=int(min(max_context, len(y_train)) if max_context else len(y_train)),
        ))

    daily_eval_df = pd.concat(daily_rows, ignore_index=True) if daily_rows else pd.DataFrame()
    cutoff_summary_df = pd.DataFrame(cutoff_rows)

    weekly_eval_df = build_aggregate_eval(daily_eval_df, context_eval, date_col, level_target_col,
                                          "week", week_days, aggregate_fun, transformation)
    monthly_eval_df = build_aggregate_eval(daily_eval_df, context_eval, date_col, level_target_col,
                                           "month", week_days, aggregate_fun, transformation)

    return dict(daily_eval=daily_eval_df, weekly_eval=weekly_eval_df, monthly_eval=monthly_eval_df,
                cutoff_summary=cutoff_summary_df, cutoff_dates=cutoff_dates)

# ============================================================
# Tablas de métricas DESAGREGADAS por punto de partida (Tarea 4)
# Excluye periodos con 29-feb (Tarea 2) y periodos sin observación completa.
# ============================================================
def disaggregated_scores(eval_df, by_offset=True):
    if eval_df is None or len(eval_df) == 0:
        return pd.DataFrame()
    d = eval_df[eval_df["has_full_observed_period"] & (~eval_df["has_leap_day"]) &
                np.isfinite(eval_df["real_proc"]) & np.isfinite(eval_df["pred_proc"])].copy()
    if len(d) == 0:
        return pd.DataFrame()
    d["_se"] = d["error_pp"] ** 2
    grp = ["week_no", "cutoff_day", "period_offset"] if by_offset else ["week_no", "cutoff_day"]
    out = d.groupby(grp, as_index=False).agg(
        n_periodos=("error_pp", "size"), ME_pp=("error_pp", "mean"),
        MAE_pp=("abs_error_pp", "mean"), MedAE_pp=("abs_error_pp", "median"),
        _mse=("_se", "mean"))
    out["RMSE_pp"] = np.sqrt(out["_mse"]); out = out.drop(columns="_mse")
    sort_cols = ["week_no", "period_offset"] if by_offset else ["week_no", "cutoff_day"]
    return out.sort_values(sort_cols).reset_index(drop=True)

# ============================================================
# Resumen mensual desagregado (punto de partida x horizonte)
# ============================================================
def horizon_labels_for(next_full_months):
    return ["Nowcast (M)"] + [f"Forecast (M+{i})" for i in range(1, next_full_months + 1)]

def score_monthly(res, next_full_months):
    me = res["monthly_eval"]
    if me is None or len(me) == 0:
        return pd.DataFrame()
    labels = horizon_labels_for(next_full_months)
    s = me[me["has_full_observed_period"] & (~me["has_leap_day"]) &
           np.isfinite(me["real_proc"]) & np.isfinite(me["pred_proc"]) &
           (me["period_offset"] >= 0) & (me["period_offset"] <= next_full_months)].copy()
    if len(s) == 0:
        return pd.DataFrame()
    s["Horizonte"] = s["period_offset"].map(lambda o: labels[o])
    s["_se"] = s["error_pp"] ** 2
    out = s.groupby(["week_no", "cutoff_day", "period_offset", "Horizonte"], as_index=False).agg(
        n=("error_pp", "size"), ME_pp=("error_pp", "mean"), MAE_pp=("abs_error_pp", "mean"),
        MedAE_pp=("abs_error_pp", "median"), _mse=("_se", "mean"))
    out["RMSE_pp"] = np.sqrt(out["_mse"]); out = out.drop(columns="_mse")
    return out.sort_values(["week_no", "period_offset"]).reset_index(drop=True)

# ============================================================
# Exportar un escenario a su propio XLSX (hojas: metadata | predictions | statistics)
# ============================================================
def export_scenario_xlsx(res, calendar, weekend_treat, target_name, transformation,
                         aggregate_fun, model_name, next_full_months, output_xlsx):
    cs = res["cutoff_summary"]; de = res["daily_eval"]; me = res["monthly_eval"]
    meta_rows, pred_cols, w = [], [], 0

    for k in range(len(cs)):
        w += 1
        cutoff_date = pd.Timestamp(cs["cutoff_date"].iloc[k])
        id_v = f"Corte {cutoff_date.date()}"

        dwin = de.loc[de["id_ventana"] == id_v, ["fecha", "y_true_level", "yhat_level"]].copy()
        dwin.columns = ["fecha", f"w{w}_true", f"w{w}_pred"]
        pred_cols.append(dwin)

        mwin = me[(me["id_ventana"] == id_v) & (me["period_offset"] >= 0) &
                  (me["period_offset"] <= next_full_months)].sort_values("period_offset")
        for _, mr in mwin.iterrows():
            meta_rows.append(dict(
                window=w, model=model_name, calendar=calendar, weekend_treat=weekend_treat,
                target=target_name, transformation=transformation, aggregate_fun=aggregate_fun,
                cutoff_week=int(cs["week_no"].iloc[k]),
                cutoff_date=cutoff_date.strftime("%Y-%m-%d"),
                pred_month=int(mr["period_offset"]),
                train_start=pd.Timestamp(cs["train_start"].iloc[k]).strftime("%Y-%m-%d"),
                test_start=pd.Timestamp(mr["period_start"]).strftime("%Y-%m-%d"),
                test_end=pd.Timestamp(mr["period_end"]).strftime("%Y-%m-%d"),
                horizon_start=pd.Timestamp(cs["forecast_start"].iloc[k]).strftime("%Y-%m-%d"),
                horizon_end=pd.Timestamp(cs["forecast_end"].iloc[k]).strftime("%Y-%m-%d"),
                real_proc=mr["real_proc"], pred_proc=mr["pred_proc"],
                abs_error_pp=mr["abs_error_pp"], has_leap_day=bool(mr["has_leap_day"]),
                full_month_obs=bool(mr["has_full_observed_period"]),
            ))

    meta_df = pd.DataFrame(meta_rows)
    pred_df = pred_cols[0]
    for nxt in pred_cols[1:]:
        pred_df = pred_df.merge(nxt, on="fecha", how="outer")
    pred_df = pred_df.sort_values("fecha").reset_index(drop=True)
    pred_df["fecha"] = pd.to_datetime(pred_df["fecha"]).dt.strftime("%Y-%m-%d")

    s = me[me["has_full_observed_period"] & (~me["has_leap_day"]) &
           np.isfinite(me["real_proc"]) & np.isfinite(me["pred_proc"]) &
           (me["period_offset"] >= 0) & (me["period_offset"] <= next_full_months)].copy()
    if len(s) > 0:
        s["_se"] = s["error_pp"] ** 2
        stats_df = s.groupby(["week_no", "period_offset"], as_index=False).agg(
            n_periodos=("error_pp", "size"), ME_pp=("error_pp", "mean"),
            MAE_pp=("abs_error_pp", "mean"), MedAE_pp=("abs_error_pp", "median"),
            _mse=("_se", "mean"))
        stats_df["RMSE_pp"] = np.sqrt(stats_df["_mse"]); stats_df = stats_df.drop(columns="_mse")
        stats_df = stats_df.rename(columns={"week_no": "cutoff_week", "period_offset": "pred_month"})
        stats_df.insert(0, "weekend_treat", weekend_treat)
        stats_df.insert(0, "calendar", calendar)
        stats_df.insert(0, "model", model_name)
    else:
        stats_df = pd.DataFrame()

    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as xw:
        meta_df.to_excel(xw, sheet_name="metadata", index=False)
        pred_df.to_excel(xw, sheet_name="predictions", index=False)
        stats_df.to_excel(xw, sheet_name="statistics", index=False)
    return output_xlsx, meta_df, pred_df, stats_df

# ============================================================
# Gráficos (matplotlib)
# ============================================================
def plot_daily(res, titulo):
    import matplotlib.pyplot as plt
    de = res["daily_eval"].copy()
    if len(de) == 0:
        print("Sin datos diarios."); return
    de["fecha"] = pd.to_datetime(de["fecha"])
    ventanas = list(pd.unique(de["id_ventana"]))
    ncol = 2; nrow = int(np.ceil(len(ventanas) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.2 * nrow), squeeze=False)
    for ax in axes.flat:
        ax.set_visible(False)
    for i, v in enumerate(ventanas):
        ax = axes.flat[i]; ax.set_visible(True)
        g = de[de["id_ventana"] == v].sort_values("fecha")
        ax.plot(g["fecha"], g["y_true_proc"], color="black", lw=0.9, label="Real")
        ax.plot(g["fecha"], g["yhat_proc"], color="royalblue", lw=0.9, ls="--", label="Forecast")
        ax.set_title(v, fontsize=9); ax.tick_params(labelsize=7)
    axes.flat[0].legend(loc="upper left", fontsize=8)
    fig.suptitle(f"{titulo}\nPredicción diaria (escala log) por ventana de corte", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

def plot_monthly(res, titulo):
    import matplotlib.pyplot as plt
    me = res["monthly_eval"]
    d = me[me["has_full_observed_period"] & (~me["has_leap_day"]) &
           np.isfinite(me["real_proc"]) & np.isfinite(me["pred_proc"])].copy()
    if len(d) == 0:
        print("Sin meses completos observados."); return
    d["Escenario"] = "Semana " + d["week_no"].astype(str) + " (corte " + d["cutoff_day"].astype(str) + ")"
    escenarios = list(pd.unique(d["Escenario"]))
    ncol = 2; nrow = int(np.ceil(len(escenarios) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.4 * nrow), squeeze=False)
    for ax in axes.flat:
        ax.set_visible(False)
    for i, e in enumerate(escenarios):
        ax = axes.flat[i]; ax.set_visible(True)
        g = d[d["Escenario"] == e].sort_values("period_start")
        ax.plot(g["period_start"], g["real_proc"], color="black", marker="o", lw=1.1, label="Real")
        ax.plot(g["period_start"], g["pred_proc"], color="royalblue", marker="o", ls="--", lw=1.1, label="Forecast")
        ax.set_title(e, fontsize=9); ax.tick_params(labelsize=7)
    axes.flat[0].legend(loc="upper left", fontsize=8)
    fig.suptitle(f"{titulo}\nAcumulado mensual (escala log) real vs forecast", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

# ============================================================
# Preparación del dato de entrada (desde Google Sheets o CSV)
# Devuelve un DataFrame con: timestamp, target, target_proc + covariables.
# ============================================================
def coerce_numeric(s):
    s = pd.Series(s).astype(str).str.strip()
    out = pd.to_numeric(s, errors="coerce")
    if out.notna().sum() == 0 or out.isna().mean() > 0.5:   # reintento con coma decimal
        out = pd.to_numeric(s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
                            errors="coerce")
    return out

def prepare_data(df_raw, transformation="log", date_col_in="timestamp",
                 target_col_in="target", dayfirst=True):
    df = df_raw.copy()
    df.columns = [str(c).strip() for c in df.columns]
    if date_col_in not in df.columns or target_col_in not in df.columns:
        raise ValueError(f"Se esperan columnas '{date_col_in}' y '{target_col_in}'. "
                         f"Encontradas: {list(df.columns)}")
    out = pd.DataFrame({
        "timestamp": pd.to_datetime(df[date_col_in], dayfirst=dayfirst, errors="coerce"),
        "target": coerce_numeric(df[target_col_in]),
    })
    out["target_proc"] = transform_level(out["target"], transformation)
    out = add_calendar_covariates(out, "timestamp").sort_values("timestamp")
    out = out.dropna(subset=["timestamp", "target", "target_proc"]).reset_index(drop=True)
    return out

## (3) Carga del modelo Toto-2 [TBC]

In [ ]:
# ============================================================
# TBC: tamaño del modelo Toto-2. Cambia TOTO_MODEL_SIZE para probar otras
# variantes: a más parámetros, mejor calidad pero más cómputo/memoria.
# ============================================================
TOTO_MODEL_SIZE   = "4m"     # "4m" | "22m" | "313m" | "1B" | "2.5B"
TOTO_DECODE_BLOCK = None      # None = inferencia rápida; 768 = mayor estabilidad a horizontes largos
TOTO_MAX_CONTEXT  = 4096      # nº máx. de días de historia que se pasan al modelo

CHECKPOINT = f"Datadog/Toto-2.0-{TOTO_MODEL_SIZE}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = Toto2Model.from_pretrained(CHECKPOINT, map_location=DEVICE).to(DEVICE).eval()
MODEL_NAME = f"Toto-2.0-{TOTO_MODEL_SIZE}"

print(f"Modelo: {CHECKPOINT} | device: {DEVICE} | "
      f"{sum(p.numel() for p in model.parameters()):,} parámetros")

## (4) Preparación del dato [TBC]

In [ ]:
### TBC: origen del dato, nombre del target y método de transformación (log / log1p / level)
TARGET_NAME    = "Consumo_BD"
TRANSFORMATION = "log"          # log / log1p / level

DATA_SOURCE = "gsheet"          # "gsheet" (Colab) | "csv" (local / offline)

# --- Google Sheets (Colab) ---
FILE_ID    = "1hakzOf9fUA8dA9TLxeph4wrd3EpWUWS0jI8QbzfGBsg"   # id del spreadsheet
SHEET1     = "Consumo_BD"                                      # nombre de la hoja
HEADER_ROW = 0                  # fila (0-index) con las cabeceras item_id/timestamp/target
                                # (el export web de BBVA Research suele usar HEADER_ROW = 2)
DAYFIRST   = True               # True si las fechas son dd/mm/aaaa

# --- CSV local (fallback offline) ---
CSV_PATH     = "data/consumo_bd.csv"
CSV_SKIPROWS = 2

In [ ]:
# ============================================================
# Carga del dato. En Colab se lee de Google Sheets con la cuenta autenticada
# (el R de Colab no puede autenticarse contra Drive; aquí en Python sí).
# ============================================================
if DATA_SOURCE == "gsheet":
    from google.colab import auth
    auth.authenticate_user()
    from google.auth import default
    creds, _ = default()

    import gspread
    gc = gspread.authorize(creds)
    spreadsheet = gc.open_by_key(FILE_ID)
    sheet = spreadsheet.worksheet(SHEET1)
    data = sheet.get_all_values()
    df_raw = pd.DataFrame(data[HEADER_ROW + 1:], columns=data[HEADER_ROW])
else:
    df_raw = pd.read_csv(CSV_PATH, skiprows=CSV_SKIPROWS, dtype=str)

Data = prepare_data(df_raw, transformation=TRANSFORMATION, dayfirst=DAYFIRST)
print("Filas:", len(Data), " rango:", Data["timestamp"].min().date(), "a", Data["timestamp"].max().date())
Data.head()

## (5) Configuración del experimento [TBC]

In [ ]:
# ============================================================
# Configuración del experimento
# ============================================================
DATE_COLUMN, TARGET_COLUMN, LEVEL_TARGET_COLUMN = "timestamp", "target_proc", "target"

TRAIN_HISTORY_START = "2017-01-01"
FORECAST_START_DATE = "2022-01-01"
LAST_CUTOFF_DATE    = "2025-12-21"
N_FORECAST_WINDOWS  = 2            # TBC: nº de ventanas (cortes). Sube para el experimento completo.
NEXT_FULL_MONTHS    = 3
AGGREGATE_FUN       = "sum"        # "sum" (flujos: consumo/ventas) | "mean" (stocks/ratios/índices)

def run_scenario(week_days, weekend_fill):
    cutoff_days = [5, 10, 15, 20] if week_days == 5 else [7, 14, 21, 28]
    return run_growing_daily_forecast(
        context_df=Data, model=model, target_col=TARGET_COLUMN, date_col=DATE_COLUMN,
        train_history_start=TRAIN_HISTORY_START, forecast_start_date=FORECAST_START_DATE,
        last_cutoff_date=LAST_CUTOFF_DATE, cutoff_days=cutoff_days, next_full_months=NEXT_FULL_MONTHS,
        week_days=week_days, weekend_fill=weekend_fill, level_target_col=LEVEL_TARGET_COLUMN,
        n_forecast_windows=N_FORECAST_WINDOWS, aggregate_fun=AGGREGATE_FUN, transformation=TRANSFORMATION,
        device=DEVICE, max_context=TOTO_MAX_CONTEXT, decode_block_size=TOTO_DECODE_BLOCK)

## (6) Ejecutar experimento: **Escenario (a) Calendario completo — 7 días**

In [ ]:
CALENDAR      = "natural_7d"
WEEKEND_TREAT = "None"
OUTPUT_XLSX   = "toto2_univariate_7dweek.xlsx"

In [ ]:
res_a = run_scenario(week_days=7, weekend_fill=None)

print("=== (a) FULL 7-DÍAS — Resumen de cortes ===")
display(res_a["cutoff_summary"][["cutoff_date", "cutoff_day", "week_no",
                                 "prediction_length", "train_rows", "week_days", "weekend_fill"]])
print("=== (a) SCORE MENSUAL DESAGREGADO (punto de partida x horizonte) ===")
score_a = score_monthly(res_a, NEXT_FULL_MONTHS); display(score_a)

In [ ]:
plot_daily(res_a,   "(a) Calendario completo 7 días")
plot_monthly(res_a, "(a) Calendario completo 7 días")

In [ ]:
# Guardar resultados del escenario (a) en su propio XLSX (metadata | predictions | statistics)
export_scenario_xlsx(res_a, calendar=CALENDAR, weekend_treat=WEEKEND_TREAT, target_name=TARGET_NAME,
                     transformation=TRANSFORMATION, aggregate_fun=AGGREGATE_FUN, model_name=MODEL_NAME,
                     next_full_months=NEXT_FULL_MONTHS, output_xlsx=OUTPUT_XLSX)
print("Guardado:", OUTPUT_XLSX)

## (6) Ejecutar experimento: **Escenario (b) Calendario de negocio — 5 días, relleno = media de la semana anterior**

In [ ]:
CALENDAR      = "business_5d"
WEEKEND_TREAT = "week_avg"
OUTPUT_XLSX   = "toto2_univariate_5dweek_mean.xlsx"

In [ ]:
res_b = run_scenario(week_days=5, weekend_fill="mean")

print("=== (b) BUSINESS 5-DÍAS (mean) — Resumen de cortes ===")
display(res_b["cutoff_summary"][["cutoff_date", "cutoff_day", "week_no",
                                 "prediction_length", "train_rows", "week_days", "weekend_fill"]])
print("=== (b) SCORE MENSUAL DESAGREGADO (punto de partida x horizonte) ===")
score_b = score_monthly(res_b, NEXT_FULL_MONTHS); display(score_b)

In [ ]:
plot_daily(res_b,   "(b) Negocio 5 días — relleno media semana anterior")
plot_monthly(res_b, "(b) Negocio 5 días — relleno media semana anterior")

In [ ]:
# Guardar resultados del escenario (b) en su propio XLSX (metadata | predictions | statistics)
export_scenario_xlsx(res_b, calendar=CALENDAR, weekend_treat=WEEKEND_TREAT, target_name=TARGET_NAME,
                     transformation=TRANSFORMATION, aggregate_fun=AGGREGATE_FUN, model_name=MODEL_NAME,
                     next_full_months=NEXT_FULL_MONTHS, output_xlsx=OUTPUT_XLSX)
print("Guardado:", OUTPUT_XLSX)

## (6) Ejecutar experimento: **Escenario (c) Calendario de negocio — 5 días, relleno = interpolación lineal**

In [ ]:
CALENDAR      = "business_5d"
WEEKEND_TREAT = "interp"
OUTPUT_XLSX   = "toto2_univariate_5dweek_lininterp.xlsx"

In [ ]:
res_c = run_scenario(week_days=5, weekend_fill="interp")

print("=== (c) BUSINESS 5-DÍAS (interp) — Resumen de cortes ===")
display(res_c["cutoff_summary"][["cutoff_date", "cutoff_day", "week_no",
                                 "prediction_length", "train_rows", "week_days", "weekend_fill"]])
print("=== (c) SCORE MENSUAL DESAGREGADO (punto de partida x horizonte) ===")
score_c = score_monthly(res_c, NEXT_FULL_MONTHS); display(score_c)

In [ ]:
plot_daily(res_c,   "(c) Negocio 5 días — relleno interpolación lineal")
plot_monthly(res_c, "(c) Negocio 5 días — relleno interpolación lineal")

In [ ]:
# Guardar resultados del escenario (c) en su propio XLSX (metadata | predictions | statistics)
export_scenario_xlsx(res_c, calendar=CALENDAR, weekend_treat=WEEKEND_TREAT, target_name=TARGET_NAME,
                     transformation=TRANSFORMATION, aggregate_fun=AGGREGATE_FUN, model_name=MODEL_NAME,
                     next_full_months=NEXT_FULL_MONTHS, output_xlsx=OUTPUT_XLSX)
print("Guardado:", OUTPUT_XLSX)

## (7) Comparativa de los tres escenarios

MAE medio (pp) por horizonte y escenario, y gráfico comparativo.

In [ ]:
comp = pd.concat([
    score_a.assign(Escenario="(a) Full 7d"),
    score_b.assign(Escenario="(b) Business 5d - mean"),
    score_c.assign(Escenario="(c) Business 5d - interp"),
], ignore_index=True)

# MAE medio por horizonte y escenario
resumen = (comp.groupby(["Escenario", "Horizonte"], as_index=False)
               .agg(MAE_pp=("MAE_pp", "mean"), MedAE_pp=("MedAE_pp", "mean"), RMSE_pp=("RMSE_pp", "mean")))
print("=== MAE MEDIO (pp) POR ESCENARIO x HORIZONTE ===")
orden = ["Nowcast (M)"] + [f"Forecast (M+{i})" for i in range(1, NEXT_FULL_MONTHS + 1)]
tabla = (resumen.pivot(index="Escenario", columns="Horizonte", values="MAE_pp")
                .reindex(columns=[h for h in orden if h in resumen["Horizonte"].unique()]))
display(tabla.round(3))

# Gráfico de barras agrupadas
escenarios = list(resumen["Escenario"].unique())
horizontes = [h for h in orden if h in resumen["Horizonte"].unique()]
x = np.arange(len(horizontes)); bw = 0.8 / max(1, len(escenarios))
fig, ax = plt.subplots(figsize=(11, 5))
for i, e in enumerate(escenarios):
    vals = [resumen[(resumen["Escenario"] == e) & (resumen["Horizonte"] == h)]["MAE_pp"].mean() for h in horizontes]
    ax.bar(x + (i - (len(escenarios) - 1) / 2) * bw, vals, width=bw, label=e)
ax.set_xticks(x); ax.set_xticklabels(horizontes)
ax.set_ylabel("MAE medio (pp)"); ax.set_xlabel("Horizonte")
ax.set_title(f"MAE medio (pp) por horizonte y escenario de calendario — {MODEL_NAME}")
ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
score_a

In [ ]:
score_b

In [ ]:
score_c